In [ ]:
# ============================================================
# CELL 0 — INSTALL (run once)
# ============================================================
!pip install -q xgboost lightgbm scikit-learn shap pandas matplotlib numpy scipy torch

# ============================================================
# === SECTION: IMPORTS ===
# ============================================================
import os, gc, math, time, copy, random, warnings, threading
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (GradientBoostingRegressor,
                               RandomForestRegressor, StackingRegressor)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
from lightgbm import LGBMRegressor
import shap

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ============================================================
# === SECTION: GPU / SEED / GLOBAL LOCKS ===
# ============================================================
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP  = torch.cuda.is_available()
HAS_COMP = int(torch.__version__.split(".")[0]) >= 2

cudnn.benchmark  = True
cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True

_PRINT_LOCK  = threading.Lock()
_SCALER_LOCK = threading.Lock()
_RESULT_LOCK = threading.Lock()

def seed_everything(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything()
print(f"[DEVICE] {DEVICE}  AMP={USE_AMP}  compile={HAS_COMP}")
if torch.cuda.is_available():
    print(f"[GPU] {torch.cuda.get_device_name(0)}")
    print(f"[GPU] VRAM = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ============================================================
# === SECTION: CONFIG ===
# ============================================================
CONFIG = {
    "seed": 42,
    "train_ratio": 0.70, "val_ratio": 0.10, "test_ratio": 0.20,
    "sigma_km": 20.0, "lags": [1,2,3,6,12,24],

    # ── Graph ──────────────────────────────────────────────
    "T": 24,
    "graph_batch_size": 1024,
    "graph_hidden_dim": 192,
    "graph_gcn_dim": 128,
    "graph_num_gcn_layers": 3,
    "graph_num_epochs": 60,
    "graph_lr": 3e-4,
    "graph_weight_decay": 1e-5,
    "graph_patience": 10,
    "graph_num_workers": 2,
    "graph_loss": "huber",
    "grad_clip": 1.0,

    # ── Parallelism ────────────────────────────────────────
    # Tier-model parallel: 4 base models × 12 nodes = 48 concurrent workers
    "node_threads": 12,    # parallel nodes per model-tier
    "model_threads": 6,    # parallel model tiers (XGB, LGBM, GBR, RF, Stack, Graph)

    # ── Output ────────────────────────────────────────────
    "dpi": 180,

    # ── Stations ──────────────────────────────────────────
    "stations": [
        "Aotizhongxin","Changping","Dingling","Dongsi",
        "Guanyuan","Gucheng","Huairou","Nongzhanguan",
        "Shunyi","Tiantan","Wanliu","Wanshouxigong"
    ],
    "coords": {
        "Aotizhongxin":  (39.982,116.407), "Changping":    (40.217,116.231),
        "Dingling":      (40.292,116.220),  "Dongsi":       (39.929,116.417),
        "Guanyuan":      (39.933,116.339),  "Gucheng":      (39.914,116.184),
        "Huairou":       (40.357,116.631),  "Nongzhanguan": (39.937,116.461),
        "Shunyi":        (40.127,116.655),  "Tiantan":      (39.886,116.407),
        "Wanliu":        (39.987,116.295),  "Wanshouxigong":(39.878,116.352),
    },

    # ── Model hyperparams ─────────────────────────────────
    "xgb_params": dict(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0, tree_method="hist",
        verbosity=0, n_jobs=2, random_state=42),
    "lgbm_params": dict(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0,
        n_jobs=2, verbose=-1, random_state=42),
    "gbr_params": dict(
        n_estimators=300, max_depth=5, learning_rate=0.03,
        subsample=0.8, min_samples_leaf=10, random_state=42),
    "rf_params": dict(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        n_jobs=-1, random_state=42),
    "ridge_alpha": 1.0, "cv_folds": 5,
}

STATIONS   = CONFIG["stations"]
N_NODES    = len(STATIONS)
ALL_MODELS = ["XGBoost","LightGBM","GradientBoosting","RandomForest","ST-Graph","Stack"]
COLORS = {
    "XGBoost":"#ef476f","LightGBM":"#118ab2","GradientBoosting":"#06d6a0",
    "RandomForest":"#ffd166","ST-Graph":"#8338ec","Stack":"#3a86ff"
}
BG="#0f172a"; PANEL="#111827"; GRID="#334155"; TXT="#e5e7eb"; MUTED="#94a3b8"
WD_MAP = {
    "N":0.,"NNE":22.5,"NE":45.,"ENE":67.5,"E":90.,"ESE":112.5,
    "SE":135.,"SSE":157.5,"S":180.,"SSW":202.5,"SW":225.,"WSW":247.5,
    "W":270.,"WNW":292.5,"NW":315.,"NNW":337.5
}
GRAPH_FEATS = ["PM2.5","PM10","SO2","NO2","CO","O3","wind_u","wind_v","TEMP","PRES","DEWP","RAIN"]

# ============================================================
# === SECTION: HELPERS ===
# ============================================================
def fname(st): return f"PRSA_Data_{st}_20130301-20170228.csv"

def haversine(la1,lo1,la2,lo2):
    R=6371.; p1,p2=math.radians(la1),math.radians(la2)
    dl=math.radians(lo2-lo1); dp=math.radians(la2-la1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def gaussian_adj():
    A=np.zeros((N_NODES,N_NODES),np.float32)
    for i,s1 in enumerate(STATIONS):
        for j,s2 in enumerate(STATIONS):
            d=haversine(*CONFIG["coords"][s1],*CONFIG["coords"][s2])
            A[i,j]=np.exp(-d**2/(2*CONFIG["sigma_km"]**2))
    return A

def sym_norm(A):
    D=np.diag(1./np.sqrt(np.maximum(A.sum(1),1e-8)))
    return (D@A@D).astype(np.float32)

def metrics(yt,yp):
    yt=np.asarray(yt,np.float64); yp=np.asarray(yp,np.float64)
    m=~(np.isnan(yt)|np.isnan(yp)); yt,yp=yt[m],yp[m]
    return (mean_absolute_error(yt,yp),
            np.sqrt(mean_squared_error(yt,yp)),
            r2_score(yt,yp),
            np.mean(np.abs((yt-yp)/np.maximum(np.abs(yt),1.)))*100.)

def style_ax(ax):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.tick_params(colors=MUTED); ax.grid(True,alpha=.15,color=GRID)
    ax.title.set_color(TXT); ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)

def plog(msg):
    with _PRINT_LOCK: print(msg)

# ============================================================
# === SECTION: DATA LOADING ===
# ============================================================
def validate_files():
    miss=[fname(st) for st in STATIONS if not os.path.exists(fname(st))]
    if miss: raise FileNotFoundError("Missing:\n"+"\n".join(miss))

def load_station(st):
    REQD=["year","month","day","hour","PM2.5","PM10","SO2","NO2","CO","O3",
          "TEMP","PRES","DEWP","RAIN","wd","WSPM"]
    df=pd.read_csv(fname(st),na_values=["NA","NaN","","  "])
    df.columns=[c.strip() for c in df.columns]
    miss=[c for c in REQD if c not in df.columns]
    if miss: raise ValueError(f"{st} missing: {miss}")
    df["datetime"]=pd.to_datetime(
        dict(year=df.year,month=df.month,day=df.day,hour=df.hour),errors="coerce")
    for c in ["PM2.5","PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM"]:
        df[c]=pd.to_numeric(df[c],errors="coerce")
    wd_deg=df["wd"].astype(str).str.strip().map(WD_MAP).astype(float)
    wr=np.deg2rad(wd_deg)
    df["wind_u"]=df["WSPM"]*np.cos(wr); df["wind_v"]=df["WSPM"]*np.sin(wr)
    keep=["datetime"]+GRAPH_FEATS
    sub=df[keep].copy()
    sub.columns=["datetime"]+[f"{st}_{c}" for c in GRAPH_FEATS]
    return sub.sort_values("datetime").reset_index(drop=True)

def load_all():
    merged=None
    for st in STATIONS:
        plog(f"  [LOAD] {fname(st)}")
        s=load_station(st)
        merged=s if merged is None else merged.merge(s,on="datetime",how="inner")
    return merged.sort_values("datetime").reset_index(drop=True)

def add_time_feats(df):
    h=df["datetime"].dt.hour; mo=df["datetime"].dt.month
    df["hour_sin"]=np.sin(2*np.pi*h/24); df["hour_cos"]=np.cos(2*np.pi*h/24)
    df["month_sin"]=np.sin(2*np.pi*mo/12); df["month_cos"]=np.cos(2*np.pi*mo/12)
    return df

def add_lags(df):
    for st in STATIONS:
        for lag in CONFIG["lags"]:
            df[f"{st}_PM2.5_lag{lag}"]=df[f"{st}_PM2.5"].shift(lag)
    return df

def add_spatial_feats(df,A):
    pm=df[[f"{st}_PM2.5" for st in STATIONS]].values.astype(np.float32)
    for i,st in enumerate(STATIONS):
        w=A[i].copy(); w[i]=0.; denom=max(w.sum(),1e-8)
        df[f"{st}_nbr_avg"]=(pm*w[None,:]).sum(1)/denom
        top3=[j for j in np.argsort(-A[i]) if j!=i][:3]
        df[f"{st}_nbr_max"]=pm[:,top3].max(1)
    return df

def clean(df):
    df=df.ffill().bfill()
    for c in df.columns:
        if c!="datetime" and df[c].isna().any():
            df[c]=df[c].fillna(df[c].median())
    return df.dropna(subset=[f"{st}_PM2.5" for st in STATIONS]).reset_index(drop=True)

# ============================================================
# === SECTION: TENSOR SCALER (GPU) ===
# ============================================================
class TensorScaler:
    def __init__(self): self.mu=None; self.sg=None
    def fit(self,X):
        with _SCALER_LOCK:
            Xt=torch.tensor(X,dtype=torch.float32,device=DEVICE)
            self.mu=Xt.mean(0,keepdim=True)
            self.sg=Xt.std(0,keepdim=True).clamp(1e-6)
        return self
    def transform_np(self,X):
        Xt=torch.tensor(X,dtype=torch.float32,device=DEVICE)
        return ((Xt-self.mu)/self.sg).cpu().numpy()

# ============================================================
# === SECTION: TABULAR FEATURES ===
# ============================================================
def tabular_node_data(df,st):
    lag_cols=[f"{st}_PM2.5_lag{l}" for l in CONFIG["lags"]]
    base=[f"{st}_{f}" for f in GRAPH_FEATS if f!="PM2.5"]
    extra=["hour_sin","hour_cos","month_sin","month_cos",
           f"{st}_nbr_avg",f"{st}_nbr_max"]
    feat_cols=lag_cols+base+extra
    return df[feat_cols].copy(), df[f"{st}_PM2.5"].copy(), feat_cols

def chrono_split(X,y):
    n=len(X)
    n_tr=int(n*CONFIG["train_ratio"]); n_va=int(n*CONFIG["val_ratio"])
    return (X.iloc[:n_tr], X.iloc[n_tr:n_tr+n_va], X.iloc[n_tr+n_va:],
            y.values[:n_tr], y.values[n_tr:n_tr+n_va], y.values[n_tr+n_va:])

# ============================================================
# === SECTION: PER-MODEL TIER PARALLEL TRAINERS ===
# Each function trains ONE model type across ALL 12 nodes in parallel.
# All 6 functions are then submitted simultaneously to a top-level executor.
# ============================================================

def _scale_node(df, st):
    X,y,feat_cols = tabular_node_data(df,st)
    X_tr,X_va,X_te,y_tr,y_va,y_te = chrono_split(X,y)
    tsc = TensorScaler().fit(X_tr.values)
    return (tsc.transform_np(X_tr.values),
            tsc.transform_np(X_va.values),
            tsc.transform_np(X_te.values),
            y_tr, y_va, y_te, feat_cols)

# ── Tier 1: XGBoost (all 12 nodes in parallel) ─────────────
def train_all_xgb(df):
    results={}
    def _train(st):
        Xtr,Xva,Xte,ytr,yva,yte,fc = _scale_node(df,st)
        p={**CONFIG["xgb_params"]}
        if torch.cuda.is_available(): p["device"]="cuda"
        n_est=p.pop("n_estimators"); p.pop("n_jobs",None); p.pop("random_state",None)
        try:
            Dtr=xgb.QuantileDMatrix(Xtr,label=ytr)
            Dte=xgb.QuantileDMatrix(Xte)
            bst=xgb.train(p,Dtr,num_boost_round=n_est,verbose_eval=False)
            pred=bst.predict(Dte)
            sk_p={**CONFIG["xgb_params"]}
            if torch.cuda.is_available(): sk_p["device"]="cuda"
            sk=xgb.XGBRegressor(**sk_p); sk.fit(Xtr,ytr)
        except Exception as e:
            plog(f"  [XGB WARN] {st}: {e}")
            sk_p={**CONFIG["xgb_params"]}
            sk=xgb.XGBRegressor(**sk_p); sk.fit(Xtr,ytr)
            pred=sk.predict(Xte)
        mae,rmse,r2,mape=metrics(yte,pred)
        plog(f"  [XGB] {st} MAE={mae:.2f}")
        with _RESULT_LOCK:
            results[st]={"pred":pred,"y_test":yte,"metrics":(mae,rmse,r2,mape),
                         "model":sk,"Xte":Xte,"feat_cols":fc}

    with ThreadPoolExecutor(max_workers=CONFIG["node_threads"]) as ex:
        futs=[ex.submit(_train,st) for st in STATIONS]
        for f in as_completed(futs): f.result()
    return results

# ── Tier 2: LightGBM (all 12 nodes in parallel) ────────────
def train_all_lgbm(df):
    results={}
    def _train(st):
        Xtr,Xva,Xte,ytr,yva,yte,fc = _scale_node(df,st)
        p={**CONFIG["lgbm_params"]}
        if torch.cuda.is_available(): p["device"]="gpu"
        try:
            m=LGBMRegressor(**p); m.fit(Xtr,ytr)
        except Exception as e:
            plog(f"  [LGBM WARN] {st}: {e}")
            p.pop("device",None); m=LGBMRegressor(**p); m.fit(Xtr,ytr)
        pred=m.predict(Xte)
        mae,rmse,r2,mape=metrics(yte,pred)
        plog(f"  [LGBM] {st} MAE={mae:.2f}")
        with _RESULT_LOCK:
            results[st]={"pred":pred,"y_test":yte,"metrics":(mae,rmse,r2,mape)}

    with ThreadPoolExecutor(max_workers=CONFIG["node_threads"]) as ex:
        futs=[ex.submit(_train,st) for st in STATIONS]
        for f in as_completed(futs): f.result()
    return results

# ── Tier 3: GradientBoosting (all 12 nodes in parallel) ────
def train_all_gbr(df):
    results={}
    def _train(st):
        Xtr,Xva,Xte,ytr,yva,yte,fc = _scale_node(df,st)
        m=GradientBoostingRegressor(**CONFIG["gbr_params"]); m.fit(Xtr,ytr)
        pred=m.predict(Xte)
        mae,rmse,r2,mape=metrics(yte,pred)
        plog(f"  [GBR] {st} MAE={mae:.2f}")
        with _RESULT_LOCK:
            results[st]={"pred":pred,"y_test":yte,"metrics":(mae,rmse,r2,mape)}

    with ThreadPoolExecutor(max_workers=CONFIG["node_threads"]) as ex:
        futs=[ex.submit(_train,st) for st in STATIONS]
        for f in as_completed(futs): f.result()
    return results

# ── Tier 4: RandomForest (all 12 nodes in parallel) ────────
def train_all_rf(df):
    results={}
    def _train(st):
        Xtr,Xva,Xte,ytr,yva,yte,fc = _scale_node(df,st)
        m=RandomForestRegressor(**CONFIG["rf_params"]); m.fit(Xtr,ytr)
        pred=m.predict(Xte)
        mae,rmse,r2,mape=metrics(yte,pred)
        plog(f"  [RF] {st} MAE={mae:.2f}")
        with _RESULT_LOCK:
            results[st]={"pred":pred,"y_test":yte,"metrics":(mae,rmse,r2,mape)}

    with ThreadPoolExecutor(max_workers=CONFIG["node_threads"]) as ex:
        futs=[ex.submit(_train,st) for st in STATIONS]
        for f in as_completed(futs): f.result()
    return results

# ── Stacking (all 12 nodes in parallel) ────────────────────
def train_all_stack(df):
    results={}
    def _train(st):
        Xtr,Xva,Xte,ytr,yva,yte,fc = _scale_node(df,st)
        xgb_p={**CONFIG["xgb_params"]}
        if torch.cuda.is_available(): xgb_p["device"]="cuda"
        lgbm_p={**CONFIG["lgbm_params"]}
        if torch.cuda.is_available(): lgbm_p["device"]="gpu"
        try: lgbm_e=LGBMRegressor(**lgbm_p)
        except: lgbm_p.pop("device",None); lgbm_e=LGBMRegressor(**lgbm_p)
        stack=StackingRegressor(
            estimators=[
                ("xgb",  xgb.XGBRegressor(**xgb_p)),
                ("lgbm", lgbm_e),
                ("gbr",  GradientBoostingRegressor(**CONFIG["gbr_params"])),
                ("rf",   RandomForestRegressor(**CONFIG["rf_params"])),
            ],
            final_estimator=Ridge(alpha=CONFIG["ridge_alpha"]),
            cv=CONFIG["cv_folds"], passthrough=True, n_jobs=-1
        )
        stack.fit(Xtr,ytr)
        pred=stack.predict(Xte)
        mae,rmse,r2,mape=metrics(yte,pred)
        plog(f"  [STACK] {st} MAE={mae:.2f}")
        with _RESULT_LOCK:
            results[st]={"pred":pred,"y_test":yte,"metrics":(mae,rmse,r2,mape)}

    with ThreadPoolExecutor(max_workers=CONFIG["node_threads"]) as ex:
        futs=[ex.submit(_train,st) for st in STATIONS]
        for f in as_completed(futs): f.result()
    return results

# ============================================================
# === SECTION: ST-GRAPH MODEL ===
# ============================================================
class GraphConvNorm(nn.Module):
    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.lin=nn.Linear(in_dim,out_dim,bias=False)
        self.norm=nn.LayerNorm(out_dim)
        self.res=nn.Linear(in_dim,out_dim,bias=False) if in_dim!=out_dim else nn.Identity()
        self.act=nn.GELU()
    def forward(self,x,A):
        Ax=torch.einsum("nm,btnf->btmf",A,x)
        return self.act(self.norm(self.lin(Ax)+self.res(x)))

class STGraphNet(nn.Module):
    def __init__(self,in_dim,gcn_dim,hidden_dim,num_gcn=3):
        super().__init__()
        self.gcn=nn.ModuleList(
            [GraphConvNorm(in_dim if l==0 else gcn_dim,gcn_dim) for l in range(num_gcn)])
        self.drop=nn.Dropout(0.15)
        self.gru=nn.GRU(gcn_dim,hidden_dim,num_layers=2,batch_first=True,
                        bidirectional=True,dropout=0.1)
        self.head=nn.Sequential(
            nn.LayerNorm(hidden_dim*2),
            nn.Linear(hidden_dim*2,hidden_dim), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(hidden_dim,1))
    def forward(self,x,A):
        B,T_,N,F_=x.shape; h=x
        for layer in self.gcn: h=self.drop(layer(h,A))
        h=h.permute(0,2,1,3).reshape(B*N,T_,-1)
        out,_=self.gru(h)
        return self.head(out[:,-1,:]).squeeze(-1).reshape(B,N)

class GraphDS(Dataset):
    def __init__(self,X,y,T):
        n=X.shape[0]-T; pin=torch.cuda.is_available()
        Xw=torch.from_numpy(np.stack([X[i:i+T] for i in range(n)]))
        yw=torch.from_numpy(np.stack([y[i+T]   for i in range(n)]))
        self.X=Xw.pin_memory() if pin else Xw
        self.y=yw.pin_memory() if pin else yw
    def __len__(self): return len(self.X)
    def __getitem__(self,idx): return self.X[idx],self.y[idx]

def tensor_scale_3d(Xtr,Xva,Xte):
    F_=Xtr.shape[-1]; flat=Xtr.reshape(-1,F_)
    mu=torch.tensor(flat.mean(0,keepdims=True),dtype=torch.float32,device=DEVICE)
    sg=torch.tensor(np.maximum(flat.std(0,keepdims=True),1e-6),dtype=torch.float32,device=DEVICE)
    def sc(X):
        t=torch.tensor(X,dtype=torch.float32,device=DEVICE)
        return ((t-mu)/sg).cpu().numpy()
    return sc(Xtr),sc(Xva),sc(Xte),mu,sg

def build_node_tensor(df):
    node_arrs=[df[[f"{st}_{f}" for f in GRAPH_FEATS]].values[:,None,:].astype(np.float32)
               for st in STATIONS]
    X_node=np.concatenate(node_arrs,axis=1)
    t_enc=df[["hour_sin","hour_cos","month_sin","month_cos"]].values.astype(np.float32)
    t_enc=np.repeat(t_enc[:,None,:],N_NODES,axis=1)
    return (np.concatenate([X_node,t_enc],axis=-1),
            df[[f"{st}_PM2.5" for st in STATIONS]].values.astype(np.float32))

def run_epoch(model,loader,A_t,opt=None,amp_sc=None):
    train=opt is not None
    model.train() if train else model.eval()
    crit=(nn.HuberLoss(delta=1.0) if CONFIG["graph_loss"]=="huber"
          else nn.MSELoss()).to(DEVICE)
    losses=[]; preds_list=[]; true_list=[]
    ctx=torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb,yb in loader:
            xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True)
            if train: opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred=model(xb,A_t); loss=crit(pred,yb)
            if train:
                if amp_sc and USE_AMP:
                    amp_sc.scale(loss).backward()
                    amp_sc.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(),CONFIG["grad_clip"])
                    amp_sc.step(opt); amp_sc.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(),CONFIG["grad_clip"])
                    opt.step()
            losses.append(loss.item())
            preds_list.append(pred.detach().cpu().float().numpy())
            true_list.append(yb.detach().cpu().float().numpy())
    P=np.concatenate(preds_list,0) if preds_list else np.empty((0,N_NODES))
    Y=np.concatenate(true_list, 0) if true_list  else np.empty((0,N_NODES))
    return float(np.mean(losses)) if losses else np.nan, P, Y

def train_graph_model(df):
    """Full ST-Graph training pipeline, returns node results dict."""
    X_all,y_all=build_node_tensor(df)
    F_TOTAL=X_all.shape[-1]; T=X_all.shape[0]
    n_tr=int(T*CONFIG["train_ratio"]); n_va=int(T*CONFIG["val_ratio"])
    Xg_tr,Xg_va,Xg_te=X_all[:n_tr],X_all[n_tr:n_tr+n_va],X_all[n_tr+n_va:]
    yg_tr,yg_va,yg_te=y_all[:n_tr],y_all[n_tr:n_tr+n_va],y_all[n_tr+n_va:]
    Xg_tr_s,Xg_va_s,Xg_te_s,g_mu,g_sg=tensor_scale_3d(Xg_tr,Xg_va,Xg_te)
    plog(f"[GRAPH] tensor shape: {X_all.shape}  F={F_TOTAL}")

    T_LOOK=CONFIG["T"]; pin=torch.cuda.is_available(); nw=CONFIG["graph_num_workers"]
    train_ds=GraphDS(Xg_tr_s,yg_tr,T_LOOK)
    val_ds  =GraphDS(Xg_va_s,yg_va,T_LOOK)
    test_ds =GraphDS(Xg_te_s,yg_te,T_LOOK)
    kw=dict(pin_memory=pin,num_workers=nw,persistent_workers=(nw>0),drop_last=False)
    train_loader=DataLoader(train_ds,batch_size=CONFIG["graph_batch_size"],shuffle=True,**kw)
    val_loader  =DataLoader(val_ds,  batch_size=CONFIG["graph_batch_size"],shuffle=False,**kw)
    test_loader =DataLoader(test_ds, batch_size=CONFIG["graph_batch_size"],shuffle=False,**kw)

    A_t=torch.tensor(sym_norm(gaussian_adj()),dtype=torch.float32,device=DEVICE)
    model=STGraphNet(F_TOTAL,CONFIG["graph_gcn_dim"],
                     CONFIG["graph_hidden_dim"],CONFIG["graph_num_gcn_layers"]).to(DEVICE)

    if HAS_COMP and torch.cuda.is_available():
        try: model=torch.compile(model,mode="reduce-overhead"); plog("[GRAPH] torch.compile ✓")
        except Exception as e: plog(f"[GRAPH] compile skip: {e}")

    try:
        opt=torch.optim.Adam(model.parameters(),lr=CONFIG["graph_lr"],
                             weight_decay=CONFIG["graph_weight_decay"],fused=True)
        plog("[GRAPH] Fused Adam ✓")
    except TypeError:
        opt=torch.optim.Adam(model.parameters(),lr=CONFIG["graph_lr"],
                             weight_decay=CONFIG["graph_weight_decay"])

    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=2,eta_min=1e-6)
    amp_sc=torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_state=None; best_val=np.inf; best_ep=-1; patience=0
    history={"tr":[],"va":[]}

    for ep in range(1,CONFIG["graph_num_epochs"]+1):
        tr_loss,_,_=run_epoch(model,train_loader,A_t,opt,amp_sc)
        va_loss,_,_=run_epoch(model,val_loader,  A_t)
        sched.step()
        history["tr"].append(tr_loss); history["va"].append(va_loss)
        gmem=torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
        plog(f"  [ST-GRAPH] Ep {ep:02d}/{CONFIG['graph_num_epochs']} "
             f"tr={tr_loss:.4f} va={va_loss:.4f} GPU={gmem:.2f}GB")
        if va_loss<best_val:
            best_val=va_loss; best_ep=ep; patience=0
            best_state=copy.deepcopy({k:v.cpu() for k,v in model.state_dict().items()})
        else:
            patience+=1
            if patience>=CONFIG["graph_patience"]:
                plog(f"  [ST-GRAPH] Early stop @ ep {ep}"); break

    model.load_state_dict({k:v.to(DEVICE) for k,v in best_state.items()})
    plog(f"[GRAPH] Best val={best_val:.5f} @ ep {best_ep}")
    _,graph_preds,graph_true=run_epoch(model,test_loader,A_t)
    node_results={st:metrics(graph_true[:,i],graph_preds[:,i])
                  for i,st in enumerate(STATIONS)}
    return node_results, graph_preds, graph_true, history, best_ep

# ============================================================
# === SECTION: PIPELINE — LOAD & ENGINEER ===
# ============================================================
validate_files()
A_raw=gaussian_adj(); A_norm=sym_norm(A_raw)

print("\n"+"="*80)
print("LOADING DATA")
print("="*80)
df=load_all()
for c in df.columns:
    if c!="datetime": df[c]=pd.to_numeric(df[c],errors="coerce")
df=add_time_feats(df); df=add_lags(df); df=add_spatial_feats(df,A_raw); df=clean(df)
print(f"\n[OK] df.shape = {df.shape}")
df_snapshot=df.copy()   # immutable reference for all parallel workers

# ============================================================
# === SECTION: FULL PARALLEL MODEL TRAINING ===
# 6 model tiers launched SIMULTANEOUSLY:
#   XGBoost (12 nodes)  ──┐
#   LightGBM (12 nodes) ──┤
#   GBR (12 nodes)      ──┼──► all run at the same time
#   RF (12 nodes)       ──┤
#   Stack (12 nodes)    ──┤
#   ST-Graph            ──┘
# ============================================================
print("\n"+"="*80)
print("LAUNCHING ALL MODEL TIERS IN PARALLEL")
print("="*80)

tier_fns = {
    "XGBoost":          lambda: train_all_xgb(df_snapshot),
    "LightGBM":         lambda: train_all_lgbm(df_snapshot),
    "GradientBoosting": lambda: train_all_gbr(df_snapshot),
    "RandomForest":     lambda: train_all_rf(df_snapshot),
    "Stack":            lambda: train_all_stack(df_snapshot),
    "ST-Graph":         lambda: train_graph_model(df_snapshot),
}

tier_results = {}
t_all_start  = time.time()

with ThreadPoolExecutor(max_workers=CONFIG["model_threads"]) as model_pool:
    future_to_tier = {model_pool.submit(fn): tier
                      for tier, fn in tier_fns.items()}
    for fut in as_completed(future_to_tier):
        tier = future_to_tier[fut]
        try:
            result = fut.result()
            tier_results[tier] = result
            plog(f"\n[✓] {tier} COMPLETE")
        except Exception as e:
            plog(f"\n[✗] {tier} FAILED: {e}")
            import traceback; traceback.print_exc()

print(f"\n[ALL TIERS] Total wall-clock = {time.time()-t_all_start:.1f}s")

# ── Unpack results ──────────────────────────────────────────
xgb_r   = tier_results.get("XGBoost",{})
lgbm_r  = tier_results.get("LightGBM",{})
gbr_r   = tier_results.get("GradientBoosting",{})
rf_r    = tier_results.get("RandomForest",{})
stack_r = tier_results.get("Stack",{})
graph_node_results, graph_preds, graph_true, g_history, g_best_ep = tier_results["ST-Graph"]

tabular_preds  = {st: {
    "XGBoost":         xgb_r[st]["pred"],
    "LightGBM":        lgbm_r[st]["pred"],
    "GradientBoosting":gbr_r[st]["pred"],
    "RandomForest":    rf_r[st]["pred"],
    "Stack":           stack_r[st]["pred"],
} for st in STATIONS}

tabular_truth  = {st: xgb_r[st]["y_test"] for st in STATIONS}
tabular_results= {st: {
    "XGBoost":         xgb_r[st]["metrics"],
    "LightGBM":        lgbm_r[st]["metrics"],
    "GradientBoosting":gbr_r[st]["metrics"],
    "RandomForest":    rf_r[st]["metrics"],
    "Stack":           stack_r[st]["metrics"],
} for st in STATIONS}
xgb_models    = {st: xgb_r[st]["model"]    for st in STATIONS}
xgb_Xtest     = {st: xgb_r[st]["Xte"]     for st in STATIONS}
xgb_feat_cols = {st: xgb_r[st]["feat_cols"] for st in STATIONS}

gc.collect(); torch.cuda.empty_cache()

# ============================================================
# === SECTION: EVALUATION TABLE ===
# ============================================================
rows=[]
for st in STATIONS:
    row={"Station":st}
    for m in ALL_MODELS:
        mae,rmse,r2,mape=(graph_node_results[st] if m=="ST-Graph"
                          else tabular_results[st][m])
        row.update({f"{m}_MAE":mae,f"{m}_RMSE":rmse,f"{m}_R2":r2,f"{m}_MAPE":mape})
    rows.append(row)

res_df=pd.DataFrame(rows)
avg={"Station":"AVERAGE"}
for c in res_df.columns:
    if c!="Station": avg[c]=res_df[c].mean()
res_df=pd.concat([res_df,pd.DataFrame([avg])],ignore_index=True)
res_df.to_csv("results_scoreboard.csv",index=False)
print("\n"+"="*120)
print(res_df.to_string(index=False))
print("="*120)

# ============================================================
# === SECTION: VISUALIZATION (all 10 plots) ===
# ============================================================

# 1) prediction_vs_true.png
fig,axes=plt.subplots(4,3,figsize=(22,22),dpi=CONFIG["dpi"])
fig.patch.set_facecolor(BG)
for ax,st in zip(axes.flat,STATIONS):
    style_ax(ax)
    yt=tabular_truth[st]; yp=tabular_preds[st]["Stack"]
    mae,_,r2,_=metrics(yt,yp)
    ax.plot(yt,color="#111",lw=1.2,label="True")
    ax.plot(yp,color=COLORS["Stack"],lw=1.1,label="Stack",alpha=.9)
    ax.set_title(f"{fname(st)}\nMAE={mae:.2f} | R²={r2:.4f}",fontsize=8,color=TXT,fontweight="bold")
    ax.set_xlabel("Test Step"); ax.set_ylabel("PM2.5 (µg/m³)")
h,l=axes.flat[0].get_legend_handles_labels()
fig.legend(h,l,loc="upper center",ncol=2,frameon=False,labelcolor=TXT)
fig.suptitle("Stacking: True vs Predicted PM2.5",color=TXT,fontsize=15,fontweight="bold",y=.999)
plt.tight_layout(rect=[0,0,1,.98])
plt.savefig("prediction_vs_true.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 2) graph_prediction_vs_true.png
fig,axes=plt.subplots(4,3,figsize=(22,22),dpi=CONFIG["dpi"])
fig.patch.set_facecolor(BG)
for ax,st,i in zip(axes.flat,STATIONS,range(N_NODES)):
    style_ax(ax)
    yt=graph_true[:,i]; yp=graph_preds[:,i]
    mae,_,r2,_=metrics(yt,yp)
    ax.plot(yt,color="#111",lw=1.2,label="True")
    ax.plot(yp,color=COLORS["ST-Graph"],lw=1.1,label="ST-Graph",alpha=.9)
    ax.set_title(f"{fname(st)}\nMAE={mae:.2f} | R²={r2:.4f}",fontsize=8,color=TXT,fontweight="bold")
    ax.set_xlabel("Test Step"); ax.set_ylabel("PM2.5 (µg/m³)")
h,l=axes.flat[0].get_legend_handles_labels()
fig.legend(h,l,loc="upper center",ncol=2,frameon=False,labelcolor=TXT)
fig.suptitle("ST-Graph: True vs Predicted PM2.5",color=TXT,fontsize=15,fontweight="bold",y=.999)
plt.tight_layout(rect=[0,0,1,.98])
plt.savefig("graph_prediction_vs_true.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 3) model_comparison_bar.png
fig,ax=plt.subplots(figsize=(24,10),dpi=CONFIG["dpi"])
fig.patch.set_facecolor(BG); style_ax(ax)
x=np.arange(N_NODES); w=0.13
mae_dict={m:[(graph_node_results[st][0] if m=="ST-Graph"
              else tabular_results[st][m][0]) for st in STATIONS] for m in ALL_MODELS}
for j,m in enumerate(ALL_MODELS):
    off=x+(j-(len(ALL_MODELS)-1)/2)*w
    ax.bar(off,mae_dict[m],width=w,color=COLORS[m],alpha=.9,label=m)
    ax.axhline(np.mean(mae_dict[m]),color=COLORS[m],ls="--",lw=1.2,alpha=.6)
ax.set_xticks(x); ax.set_xticklabels(STATIONS,rotation=30,ha="right",color=TXT)
ax.set_ylabel("MAE")
ax.set_title("Per-Node MAE: Five Tiers + Stacking Ensemble",color=TXT,fontsize=15,fontweight="bold")
ax.legend(facecolor=PANEL,edgecolor=GRID,labelcolor=TXT)
plt.tight_layout(); plt.savefig("model_comparison_bar.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 4) residual_analysis.png
res_cat=np.concatenate([tabular_truth[st]-tabular_preds[st]["Stack"] for st in STATIONS])
pred_cat=np.concatenate([tabular_preds[st]["Stack"] for st in STATIONS])
fig,axes=plt.subplots(2,2,figsize=(16,12),dpi=CONFIG["dpi"]); fig.patch.set_facecolor(BG)
for a in axes.flat: style_ax(a)
axes[0,0].hist(res_cat,bins=80,color=COLORS["Stack"],alpha=.85,edgecolor=BG)
axes[0,0].axvline(0,color="#f87171",ls="--",lw=1.5)
axes[0,0].set_title("Residual Histogram",color=TXT,fontweight="bold")
axes[0,1].scatter(pred_cat,res_cat,s=4,alpha=.3,color=COLORS["Stack"])
axes[0,1].axhline(0,color="#f87171",ls="--",lw=1.5)
axes[0,1].set_title("Residual vs Predicted",color=TXT,fontweight="bold")
(osm,osr),(sl,ic,_)=stats.probplot(res_cat,dist="norm")
axes[1,0].scatter(osm,osr,s=6,alpha=.35,color=COLORS["Stack"])
axes[1,0].plot(osm,sl*np.array(osm)+ic,color="#f87171",lw=1.5)
axes[1,0].set_title("Q-Q Plot (Stack)",color=TXT,fontweight="bold")
hu="Huairou"; hu_r=tabular_truth[hu]-tabular_preds[hu]["Stack"]
axes[1,1].plot(hu_r,color=COLORS["Stack"],lw=1.)
axes[1,1].axhline(0,color="#f87171",ls="--",lw=1.5)
axes[1,1].set_title("Residuals Over Time — Huairou",color=TXT,fontweight="bold")
fig.suptitle("Residual Analysis",color=TXT,fontsize=15,fontweight="bold")
plt.tight_layout(rect=[0,0,1,.97])
plt.savefig("residual_analysis.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 5) metrics_heatmap.png
MIDX={"MAE":0,"RMSE":1,"R2":2,"MAPE":3}
fig,axes=plt.subplots(1,4,figsize=(24,9),dpi=CONFIG["dpi"]); fig.patch.set_facecolor(BG)
for ax,mk in zip(axes,["MAE","RMSE","R2","MAPE"]):
    style_ax(ax); idx=MIDX[mk]
    mat=np.array([[graph_node_results[st][idx] if m=="ST-Graph"
                   else tabular_results[st][m][idx] for st in STATIONS]
                  for m in ALL_MODELS],dtype=np.float32)
    im=ax.imshow(mat,aspect="auto",cmap="viridis")
    ax.set_title(mk,color=TXT,fontweight="bold")
    ax.set_yticks(np.arange(len(ALL_MODELS))); ax.set_yticklabels(ALL_MODELS,color=TXT)
    ax.set_xticks(np.arange(N_NODES))
    ax.set_xticklabels(STATIONS,rotation=45,ha="right",color=TXT,fontsize=7)
    cb=plt.colorbar(im,ax=ax,fraction=.046,pad=.04); cb.ax.tick_params(colors=MUTED)
fig.suptitle("Metrics Heatmap: Stations × Models",color=TXT,fontsize=15,fontweight="bold")
plt.tight_layout(rect=[0,0,1,.96])
plt.savefig("metrics_heatmap.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 6 & 7) SHAP
shap_st="Aotizhongxin"
try:
    explainer=shap.TreeExplainer(xgb_models[shap_st])
    shap_X=pd.DataFrame(xgb_Xtest[shap_st],columns=xgb_feat_cols[shap_st])
    sv=explainer.shap_values(shap_X)
    for fn_s,pt in [("shap_summary.png",None),("shap_importance.png","bar")]:
        plt.figure(figsize=(12,8))
        shap.summary_plot(sv,shap_X,plot_type=pt,show=False)
        plt.title(f"SHAP {'Summary' if pt is None else 'Importance'} — {fname(shap_st)}",fontsize=12,fontweight="bold")
        plt.tight_layout(); plt.savefig(fn_s,dpi=CONFIG["dpi"],bbox_inches="tight"); plt.close()
except Exception as e:
    plog(f"[SHAP] failed: {e}")
    for fn_s in ["shap_summary.png","shap_importance.png"]:
        fig,ax=plt.subplots(figsize=(8,4)); fig.patch.set_facecolor(BG); style_ax(ax)
        ax.text(.5,.5,f"SHAP unavailable:\n{e}",ha="center",va="center",color=TXT); ax.axis("off")
        plt.tight_layout(); plt.savefig(fn_s,dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 8) improvement_bar.png
rf_mae=np.array([tabular_results[st]["RandomForest"][0] for st in STATIONS])
st_mae=np.array([tabular_results[st]["Stack"][0]        for st in STATIONS])
impr=((rf_mae-st_mae)/np.maximum(rf_mae,1e-8))*100.
fig,ax=plt.subplots(figsize=(16,7),dpi=CONFIG["dpi"]); fig.patch.set_facecolor(BG); style_ax(ax)
bcolors=["#22c55e" if v>=0 else "#ef4444" for v in impr]
bars=ax.bar(STATIONS,impr,color=bcolors,alpha=.9)
for rect,val in zip(bars,impr):
    ax.text(rect.get_x()+rect.get_width()/2,rect.get_height()+(0.5 if val>=0 else -1.2),
            f"{val:.1f}%",ha="center",fontsize=9,color=TXT)
ax.axhline(0,color=GRID,lw=1.5)
ax.set_title("Stack % MAE Improvement over RandomForest",color=TXT,fontsize=14,fontweight="bold")
ax.set_ylabel("% Improvement"); ax.set_xticklabels(STATIONS,rotation=30,ha="right",color=TXT)
plt.tight_layout(); plt.savefig("improvement_bar.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 9) graph_training_curve.png
fig,ax=plt.subplots(figsize=(10,6),dpi=CONFIG["dpi"]); fig.patch.set_facecolor(BG); style_ax(ax)
ax.plot(g_history["tr"],color="#22c55e",lw=2,label="Train Loss")
ax.plot(g_history["va"],color="#f59e0b",lw=2,label="Val Loss")
ax.axvline(g_best_ep-1,color="#f87171",ls="--",lw=1.5,label=f"Best ep={g_best_ep}")
ax.set_title("ST-Graph Training Curve",color=TXT,fontsize=14,fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Huber Loss")
ax.legend(facecolor=PANEL,edgecolor=GRID,labelcolor=TXT)
plt.tight_layout(); plt.savefig("graph_training_curve.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

# 10) per_model_summary.png
avg_mae={m:np.mean([graph_node_results[st][0] if m=="ST-Graph"
                    else tabular_results[st][m][0] for st in STATIONS]) for m in ALL_MODELS}
avg_r2 ={m:np.mean([graph_node_results[st][2] if m=="ST-Graph"
                    else tabular_results[st][m][2] for st in STATIONS]) for m in ALL_MODELS}
fig,axes=plt.subplots(1,2,figsize=(18,7),dpi=CONFIG["dpi"]); fig.patch.set_facecolor(BG)
for ax in axes: style_ax(ax)
bar_names=list(avg_mae.keys())
axes[0].barh(bar_names,[avg_mae[m] for m in bar_names],color=[COLORS[m] for m in bar_names],alpha=.9)
for i,m in enumerate(bar_names):
    axes[0].text(avg_mae[m]+.1,i,f"{avg_mae[m]:.2f}",va="center",color=TXT,fontsize=9)
axes[0].set_title("Average MAE (all 12 nodes)",color=TXT,fontweight="bold"); axes[0].tick_params(colors=TXT)
axes[1].barh(bar_names,[avg_r2[m] for m in bar_names],color=[COLORS[m] for m in bar_names],alpha=.9)
for i,m in enumerate(bar_names):
    axes[1].text(max(avg_r2[m]-.05,0),i,f"{avg_r2[m]:.4f}",va="center",color=TXT,fontsize=9)
axes[1].set_title("Average R² (all 12 nodes)",color=TXT,fontweight="bold"); axes[1].tick_params(colors=TXT)
fig.suptitle("Model Summary: Parallel Training Results",color=TXT,fontsize=15,fontweight="bold")
plt.tight_layout(); plt.savefig("per_model_summary.png",dpi=CONFIG["dpi"],bbox_inches="tight",facecolor=BG); plt.close()

print("\n[✓] All outputs saved:")
for f in ["results_scoreboard.csv","prediction_vs_true.png","graph_prediction_vs_true.png",
          "model_comparison_bar.png","residual_analysis.png","metrics_heatmap.png",
          "shap_summary.png","shap_importance.png","improvement_bar.png",
          "graph_training_curve.png","per_model_summary.png"]:
    sz=os.path.getsize(f)/1e3 if os.path.exists(f) else 0
    print(f"  {f}  ({sz:.0f} KB)")

[DEVICE] cuda  AMP=True  compile=True
[GPU] Tesla T4
[GPU] VRAM = 15.6 GB

LOADING DATA
  [LOAD] PRSA_Data_Aotizhongxin_20130301-20170228.csv
  [LOAD] PRSA_Data_Changping_20130301-20170228.csv
  [LOAD] PRSA_Data_Dingling_20130301-20170228.csv
  [LOAD] PRSA_Data_Dongsi_20130301-20170228.csv
  [LOAD] PRSA_Data_Guanyuan_20130301-20170228.csv
  [LOAD] PRSA_Data_Gucheng_20130301-20170228.csv
  [LOAD] PRSA_Data_Huairou_20130301-20170228.csv
  [LOAD] PRSA_Data_Nongzhanguan_20130301-20170228.csv
  [LOAD] PRSA_Data_Shunyi_20130301-20170228.csv
  [LOAD] PRSA_Data_Tiantan_20130301-20170228.csv
  [LOAD] PRSA_Data_Wanliu_20130301-20170228.csv
  [LOAD] PRSA_Data_Wanshouxigong_20130301-20170228.csv

[OK] df.shape = (35064, 245)

LAUNCHING ALL MODEL TIERS IN PARALLEL
[GRAPH] tensor shape: (35064, 12, 16)  F=16
[GRAPH] torch.compile ✓
[GRAPH] Fused Adam ✓
  [XGB] Dongsi MAE=7.19
  [XGB] Changping MAE=6.85
  [XGB] Aotizhongxin MAE=6.41
  [XGB] Dingling MAE=8.73
  [XGB] Guanyuan MAE=7.16
  [XGB] Wanshoux

W0613 06:19:11.633000 819 torch/_inductor/utils.py:1731] [0/0_1] Not enough SMs to use max_autotune_gemm mode


  [LGBM] Huairou MAE=6.17
  [LGBM] Nongzhanguan MAE=7.16
  [LGBM] Gucheng MAE=7.24
  [LGBM] Aotizhongxin MAE=6.26
  [LGBM] Dingling MAE=8.89
  [LGBM] Tiantan MAE=6.92
  [LGBM] Shunyi MAE=6.58
  [LGBM] Changping MAE=6.86
  [LGBM] Dongsi MAE=7.45
  [LGBM] Wanshouxigong MAE=7.37
  [LGBM] Guanyuan MAE=7.30
  [LGBM] Wanliu MAE=6.57

[✓] LightGBM COMPLETE
  [RF] Dingling MAE=8.08
  [RF] Huairou MAE=5.80
  [RF] Changping MAE=6.50
  [RF] Aotizhongxin MAE=5.57
  [RF] Gucheng MAE=6.94
  [RF] Guanyuan MAE=7.03
  [RF] Shunyi MAE=5.98
  [RF] Tiantan MAE=6.40
  [RF] Nongzhanguan MAE=6.45
  [RF] Wanshouxigong MAE=6.93
  [RF] Dongsi MAE=7.07
  [RF] Wanliu MAE=6.10

[✓] RandomForest COMPLETE
  [GBR] Dingling MAE=8.63
  [GBR] Aotizhongxin MAE=6.15
  [GBR] Changping MAE=6.93
  [GBR] Gucheng MAE=7.32
  [GBR] Huairou MAE=6.20
  [GBR] Guanyuan MAE=7.31
  [GBR] Nongzhanguan MAE=7.06
  [GBR] Dongsi MAE=7.34
  [GBR] Shunyi MAE=6.50
  [GBR] Tiantan MAE=6.76
  [GBR] Wanshouxigong MAE=7.42
  [GBR] Wanliu MAE=6.55